# CVE description to CWE: base model vs QLoRA fine-tuned model

Public data (`xamxte/cve-to-cwe`), Qwen2.5-1.5B-Instruct, 4-bit QLoRA, one T4 GPU.

Run the cells in order. The dry-run section is a short run that catches environment problems in a few minutes before the full run. Nothing in this notebook produces a number you can quote until the full run finishes and you send back the results archive from the last cell.

Runtime: Runtime > Change runtime type > T4 GPU.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Switch the runtime to a T4 GPU and run this cell again."
print(torch.cuda.get_device_name(0))
print("compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
import zipfile
from pathlib import Path

if not Path("cwe-qlora").exists():
    archives = sorted(Path(".").glob("cwe-qlora*.zip"))
    if not archives:
        from google.colab import files

        files.upload()
        archives = sorted(Path(".").glob("cwe-qlora*.zip"))
    assert archives, "Upload the cwe-qlora zip file when asked."
    with zipfile.ZipFile(archives[-1]) as bundle:
        bundle.extractall(".")
print(sorted(path.name for path in Path("cwe-qlora").iterdir()))

In [ ]:
%cd cwe-qlora
!pip install -q -U transformers peft accelerate bitsandbytes datasets huggingface_hub scikit-learn
!python -c "import torch, transformers, peft, bitsandbytes, accelerate; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__, accelerate.__version__)"

## Dry run

Tiny sizes, 20 optimizer steps. It only proves the pipeline runs on this machine (4-bit loading, fp16 on T4, training, generation). Its numbers mean nothing and must not be reported. If any cell here fails, stop and send the full error text.

In [ ]:
!python -m cwe_qlora prepare --out work_dry --run-mode dry-run --train-size 200 --val-size 50 --test-size 50
!python -m cwe_qlora train --work work_dry --output outputs_dry/adapter --max-steps 20 --eval-steps 20 --save-steps 1000 --logging-steps 5
!python -m cwe_qlora predict --work work_dry --out-dir results_dry --limit 16
!python -m cwe_qlora predict --work work_dry --out-dir results_dry --limit 16 --adapter outputs_dry/adapter

## Full run

Defaults: 12,000 training examples, 500 validation, 2,000 test, 1 epoch, effective batch 32, learning rate 2e-4, LoRA r=16 on all attention and MLP projections. If the session disconnects during training, reconnect, run the install cell again and add `--resume` to the train command.

In [ ]:
!python -m cwe_qlora prepare --out work
!python -m cwe_qlora baselines --work work --out results

In [ ]:
!python -m cwe_qlora train --work work --output outputs/adapter

In [ ]:
!python -m cwe_qlora predict --work work --out-dir results
!python -m cwe_qlora predict --work work --out-dir results --adapter outputs/adapter

In [ ]:
!python -m cwe_qlora report --work work --out results

## Package the results

Download `cwe_qlora_results.zip` and send it back unchanged. It contains the report, per-row predictions, run metadata and the training log. It does not contain model weights.

In [ ]:
import zipfile
from pathlib import Path

members = sorted(Path("results").glob("*"))
members += [Path("work/prepare_meta.json"), Path("work/train_run.json"), Path("outputs/training_log.json")]
with zipfile.ZipFile("cwe_qlora_results.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for member in members:
        bundle.write(member, member.as_posix())
print([member.as_posix() for member in members])

from google.colab import files

files.download("cwe_qlora_results.zip")